# Hyperelastic Beam — Large Deformation by Energy Minimisation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/hyperelastic_beam.ipynb)

A rubber beam twisted by a surface traction, at deformations far beyond the
linear regime. Two things make this notebook different from the
[cantilever](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/cantilever_beam.ipynb):

1. The material law is **Neo-Hookean**, defined by a strain-energy density
   rather than a bilinear form — you write $\Psi(F)$ and stop.
2. There is **no residual, no tangent stiffness and no Newton loop**. The total
   potential energy is a scalar; autograd differentiates it, and LBFGS minimises
   it directly. Boundary conditions are imposed by masking, so fixed nodes
   simply receive no gradient.

This is the shortest path from "I know the energy" to "I have the deformed
shape" that a PyTorch-native FEM library can offer.

⏱️🖥️ *Installs pyvista; the load-stepped LBFGS solve takes a few minutes on
Colab's free CPU runtime. Raise `chara_length` for a quicker run.*

Docs: [Hyperelastic Beam](https://docs.tensor-mesh.com/example_gallery/solid/hyperelastic_beam.html) · Source: [`examples/solid/hyperelastic_beam/hyperelastic_beam.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/solid/hyperelastic_beam/hyperelastic_beam.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# pyvista renders the 3D figures; xvfb gives it a virtual display to draw into.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa libgl1-mesa-glx xvfb > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0 pyvista

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Material model and load

Note `element_energy` in place of `forward`: that is how TensorMesh knows the
assembler defines an energy functional to be differentiated rather than a
matrix to be assembled.

In [ ]:
import warnings

import pyvista as pv
import vtk

vtk.vtkObject.GlobalWarningDisplayOff()          # software-OpenGL fallback notices
warnings.filterwarnings("ignore", message=".*Use vtk with osmesa.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pyvista.*")
pv.OFF_SCREEN = True

import torch

from tensormesh import Mesh
from tensormesh.assemble import ElementAssembler, FacetAssembler
from tensormesh.material import Rubber
from tensormesh.visualization import plot_deformation


class NeoHookeanModel(ElementAssembler):
    r"""Compressible Neo-Hookean strain energy density

    .. math::

        \Psi = \tfrac{\mu}{2}(I_1 - 3) - \mu \ln J + \tfrac{\lambda}{2}(\ln J)^2,

    with ``F = I + grad(u)`` the deformation gradient and ``J = det F``.
    Defining ``element_energy`` (rather than ``forward``) is what tells
    TensorMesh this is an energy functional to be differentiated, not a
    bilinear form to be assembled into a matrix.
    """

    def __post_init__(self, mu, lam):
        self.mu = mu
        self.lam = lam

    def element_energy(self, gradu):
        dim = gradu.shape[-1]
        F = torch.eye(dim, device=gradu.device, dtype=gradu.dtype) + gradu
        J = torch.clamp(torch.det(F), min=1e-6)
        logJ = torch.log(J)
        I1 = (F ** 2).sum()
        return 0.5 * self.mu * (I1 - 3) - self.mu * logJ + 0.5 * self.lam * logJ ** 2


class TorsionTraction(FacetAssembler):
    """Torsional surface traction t(x) = C (0, -dz, dy) on the end face,
    integrated over its facets into a consistent nodal load."""

    def __post_init__(self, C, y_center, z_center):
        self.C = C
        self.y_center = y_center
        self.z_center = z_center

    def forward(self, v, x):
        dy = x[1] - self.y_center
        dz = x[2] - self.z_center
        zero = torch.zeros_like(dy)
        t = torch.stack([zero, -dz * self.C, dy * self.C])
        return v[:, None] * t[None, :]

## Mesh, material, boundary conditions

In [ ]:
# 1 m x 0.4 m x 0.4 m rubber beam. Quadratic tetrahedra: large deformation on
# linear tets locks badly.
with quiet():
    mesh = Mesh.gen_cube(chara_length=0.1, order=2, left=0.0, right=1.0,
                         bottom=0.0, top=0.4, front=0.0, back=0.4)
points = mesh.points
print(f"mesh: {points.shape[0]} nodes (P2 tetrahedra)")

mu, lam = Rubber.lame_params
model = NeoHookeanModel.from_mesh(mesh, mu=mu, lam=lam)
print(f"material: {Rubber.name}  mu={mu:.2e}  lambda={lam:.2e}")

# Clamp x = 0; twist the x = 1 face with an integrated traction field.
eps = 1e-5
fixed_mask = torch.abs(points[:, 0]) < eps
traction = TorsionTraction.from_mesh(
    mesh, boundary_mask=torch.abs(points[:, 0] - 1.0) < eps,
    quadrature_order=4, C=2.4e7, y_center=0.2, z_center=0.2,
)
f_ext_final = traction().reshape(points.shape)

## Minimise the potential energy

The load is ramped in a few steps: starting LBFGS from the undeformed state at
full torque would ask it to find a strongly nonlinear minimum in one go.

In [ ]:
# Energy minimisation rather than Newton on a residual: the total potential
# energy is a scalar, autograd supplies its gradient, and LBFGS does the rest.
# Dirichlet data is imposed by masking, so fixed nodes simply receive no gradient.
u = torch.zeros_like(points, requires_grad=True)
optimizer = torch.optim.LBFGS([u], lr=1.0, max_iter=60, max_eval=80,
                              tolerance_grad=1e-5, tolerance_change=1e-5,
                              history_size=100, line_search_fn="strong_wolfe")

mask = (~fixed_mask).unsqueeze(1).to(u.dtype)
f_ext = f_ext_final.clone()


def closure():
    optimizer.zero_grad()
    u_active = u * mask
    loss = model.energy(point_data={"u": u_active}) - (f_ext * u_active).sum()
    if loss.requires_grad:
        loss.backward()
    return loss


N_STEPS = 6   # load stepping keeps the nonlinear solve well-behaved
for step in range(1, N_STEPS + 1):
    f_ext = f_ext_final * (step / N_STEPS)
    loss_val = optimizer.step(closure)
    with torch.no_grad():
        max_d = u.norm(dim=1).max().item()
    print(f"step {step}/{N_STEPS}: energy = {loss_val.item():.4e}, max disp = {max_d:.4f} m")

with torch.no_grad():
    u.data = u.data * mask
u_vec = u.detach()
print(f"\nfinal max displacement: {u_vec.norm(dim=1).max().item() * 1000:.1f} mm")

## The twisted beam

In [ ]:
plot_deformation(mesh, u_vec, "hyperelastic_rubber.png", scale_factor=1.0,
                 camera_position="isometric", fixed_nodes=fixed_mask,
                 force_vectors=f_ext_final)

from IPython.display import Image
Image("hyperelastic_rubber.png")